# Extracció de dades de l'ERP amb SQL

Distribucions Es Pla prepara l'informe de vendes de l'estiu per a la gerència. Abans de fer-lo, diversos departaments han fet arribar a l'equip de dades consultes i dubtes sobre el que hi ha a l'ERP. L'equip els ha de respondre treballant directament sobre la base de dades i, al final, deixar per escrit els criteris amb què es farà l'extracció setmanal.

## Condicions de treball

- El treball es fa en un notebook nou a JupyterHub, a partir de la connexió establerta a la sessió anterior.
- Els filtres, les transformacions i les agregacions es resolen en SQL, a la base de dades.
- Els valors variables de les consultes es passen com a paràmetres.
- Cada resposta queda documentada amb la consulta, el resultat i una justificació breu de les decisions preses.
- Cap resultat s'escriu a mà: tot s'ha de poder tornar a obtenir executant el notebook.

## 1. Abans de començar

1. Establir la connexió amb l'ERP i disposar d'una funció reutilitzable per executar consultes. Si una consulta falla, la sessió ha de poder continuar sense reiniciar el notebook.
2. Determinar fins a quina data arriben les dades de l'ERP i en quina situació són les comandes més recents. Explicar què implica per a una extracció que s'executi cada dilluns.

In [2]:
HOST = "pg-5104-distribucio.postgres.database.azure.com"
PORT = 5432
BASE_DE_DADES = "erp_distribucio"
SSLMODE = "require"          # el servidor només accepta connexions xifrades
TENANT = "politecnicllevant.cat"   # el directori Entra de l'institut

In [ ]:
USUARI = "agalmes1796@alumnes.politecnicllevant.cat"   # el vostre correu de l'institut

In [ ]:
USUARI = "agalmes1796@alumnes.politecnicllevant.cat"   # el vostre correu de l'institut

In [ ]:
import os

import psycopg
from azure.identity import DeviceCodeCredential

ESCOP = "https://ossrdbms-aad.database.windows.net/.default"   # el "recurs" de les bases de dades d'Azure


def obtenir_token():
    if os.environ.get("ERP_TOKEN"):          # només per a proves del professor
        return os.environ["ERP_TOKEN"]
    credencial = DeviceCodeCredential(tenant_id=TENANT)
    return credencial.get_token(ESCOP).token


USUARI = os.environ.get("ERP_USER", USUARI)
TOKEN = obtenir_token()


def connecta(base_de_dades=BASE_DE_DADES):
    return psycopg.connect(host=HOST, port=PORT, dbname=base_de_dades, user=USUARI,
                           password=TOKEN, sslmode=SSLMODE, connect_timeout=15, autocommit=True, keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=3)


con = connecta()
print("Connectat com a", con.execute("SELECT current_user").fetchone()[0])
print(con.execute("SELECT version()").fetchone()[0])
print("Connexió xifrada:", con.info.pgconn.ssl_in_use)

In [ ]:
# %pip install "psycopg[binary]"

In [ ]:
# import sys
# !{sys.executable} -m ensurepip --upgrade

In [ ]:
# import sys
# !{sys.executable} -m pip install --upgrade pip

In [3]:
import os

import psycopg
from azure.identity import DeviceCodeCredential

ESCOP = "https://ossrdbms-aad.database.windows.net/.default"   # el "recurs" de les bases de dades d'Azure


def obtenir_token():
    if os.environ.get("ERP_TOKEN"):          # només per a proves del professor
        return os.environ["ERP_TOKEN"]
    credencial = DeviceCodeCredential(tenant_id=TENANT)
    return credencial.get_token(ESCOP).token


USUARI = "agalmes1796"
TOKEN = "pla-5u7sm-qjv4t"


def connecta(base_de_dades=BASE_DE_DADES):
    return psycopg.connect(host=HOST, port=PORT, dbname=base_de_dades, user=USUARI,
                           password=TOKEN, sslmode=SSLMODE, connect_timeout=15, autocommit=True, keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=3)


con = connecta()
print("Connectat com a", con.execute("SELECT current_user").fetchone()[0])
print(con.execute("SELECT version()").fetchone()[0])
print("Connexió xifrada:", con.info.pgconn.ssl_in_use)

ModuleNotFoundError: No module named 'azure'

In [ ]:
import pandas as pd


def consulta(sql, params=None):
    """Executa una consulta i torna un DataFrame amb els noms de les columnes."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

In [ ]:
consulta("""
    SELECT current_database(), current_user, current_schema();
""")

In [ ]:
consulta("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'erp'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name;
""")

In [ ]:
consulta("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'erp'
      AND table_name = 'comandes'
    ORDER BY ordinal_position;
""")

In [ ]:
consulta("""
    SELECT
        data_comanda,
        data_entrega,
        estat,
        observacions
    FROM erp.comandes
    ORDER BY data_comanda DESC
    LIMIT 10;
""")

## 2. Magatzem

La responsable de magatzem organitza els torns del personal i té dubtes sobre quan arriba la feina.

1. Afirma que entren comandes abans de les 7.00 h, quan encara no hi ha ningú. **Quantes comandes del juliol de 2026 es van registrar abans d'aquesta hora, hora de Mallorca?**

In [ ]:
consulta("""
SELECT * FROM erp.comandes LIMIT 5;
""")

In [ ]:
consulta("""
    SELECT COUNT(*) AS comandes_abans_les_7
    FROM erp.comandes
    WHERE data_comanda >= TIMESTAMPTZ '2026-07-01 00:00:00 Europe/Madrid'
      AND data_comanda <  TIMESTAMPTZ '2026-08-01 00:00:00 Europe/Madrid'
      AND (data_comanda AT TIME ZONE 'Europe/Madrid')::time
          < TIME '07:00:00';
""")